In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import scipy.io as io
import sys
import numpy as np

global_path = "/home/ids/edabier/HSU"
# global_path = "/home/edabier/Documents/Thèse/benchmark"
# global_path = "/Users/edabier/Documents/Thèse/Thèse_Télécom"
sys.path.append(f"{global_path}/SS-HSU_benchmark")

from src.utils import utils, plots

if torch.cuda.is_available():
    dev = "cuda:0"
    torch.set_default_device(dev)
    print(f"Using device: {dev}")
  
else:
    dev = "cpu"
    print(f"Using device: {dev}")

Using device: cuda:0


In [2]:
dataset = "urban"
data = io.loadmat(f"{global_path}/SS-HSU_benchmark/datasets/{dataset}.mat")
Y_flat = torch.tensor(data["Y"], dtype=torch.float)
A_flat = torch.tensor(data["A"], dtype=torch.float)
E_init = torch.tensor(data["E"], dtype=torch.float)
B, c, N = E_init.shape[0], E_init.shape[1], Y_flat.shape[1]

Y_init = utils.oneD_to_2d(Y_flat)
H = Y_init.shape[-1]
A_init = utils.oneD_to_2d(A_flat)
Y_init = Y_init.unsqueeze(0)
Y_init_n = utils.standardise(Y_init)
A_init = A_init.unsqueeze(0)

if dataset == "urban4":
    wavelengths_path = f"{global_path}/SS-HSU_benchmark/datasets/urban_wavelength.txt"
else:
    wavelengths_path = f"{global_path}/SS-HSU_benchmark/datasets/{dataset}_wavelength.txt"
with open(wavelengths_path, "r") as file:
    lines = file.readlines()
    wavelengths = [float(line.strip()) for line in lines if line.strip()]

### Universat

In [3]:
universat = torch.hub.load("gastruc/UniverSat", "from_pretrained").eval()
feat = universat(x={"hsi":Y_init}, wavelengths={"hsi":wavelengths}, input_res={"hsi":30}, scale=100, latent_grid=16, output_grid=H**2, subpatches={"hsi":1})

Using cache found in /home/ids/edabier/.cache/torch/hub/gastruc_UniverSat_main
/home/ids/edabier/miniconda3/envs/hsu-latest/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.44 GiB. GPU 0 has a total capacity of 44.42 GiB of which 15.87 GiB is free. Including non-PyTorch memory, this process has 28.54 GiB memory in use. Of the allocated memory 27.91 GiB is allocated by PyTorch, and 333.18 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
plots.plot_hsi(feat[0][0].T)
plt.show()
plots.plot_pca_features(feat[0][0].T)

### Panopticon

In [ ]:
# panopticon = torch.hub.load('Panopticon-FM/panopticon','panopticon_vitb14')

Using cache found in /home/ids/edabier/.cache/torch/hub/Panopticon-FM_panopticon_main
/home/ids/edabier/.cache/torch/hub/Panopticon-FM_panopticon_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/home/ids/edabier/.cache/torch/hub/Panopticon-FM_panopticon_main/dinov2/layers/attention.py:34: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/home/ids/edabier/.cache/torch/hub/Panopticon-FM_panopticon_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
100%|██████████| 378M/378M [00:28<00:00, 14.1MB/s] 


In [ ]:
# x_dict = dict(
#   imgs = Y_init,
#   chn_ids = torch.tensor(wavelengths).repeat(1,1)
# )

# blk_indices = [11]
# features_pan = panopticon.get_intermediate_layers(x_dict, n=blk_indices, return_class_token=False)[0]
# features_pan = features_pan[0].T
# print("Alpha = ", int(features_pan.shape[0]**0.5))

# plots.plot_hsi(features_pan)
# plt.show()
# plots.plot_pca_features(features_pan)